# ⚔️ Support Vector Machine (SVM) s Bayesovskou optimalizací
**Kurz: Data Science / Machine Learning**  
**Sekce: Homework mezi dny 2 a 3 (Klasifikace)**

### Cíl cvičení:
1. Načíst standardizovaný dataset `diabetes_scaled.csv`.
2. Rozdělit data v poměru 70/30 (stratifikovaně).
3. Definovat hyperparametrický prostor pro knihovnu `hyperopt` (`kernel`, `gamma`, `C`).
4. Definovat účelovou funkci (objective function) pro křížovou validaci a metriku `f1`.
5. Spustit Bayesovskou optimalizaci (TPE – Tree of Parzen Estimators) přes `fmin`.
6. Natrénovat model `SVC` s nejlepšími parametry.
7. Otestovat efektivitu na testovací sadě a provést **velké mezimodelové srovnání** všech 3 modelů (k-NN vs. LogReg vs. SVM).


In [ ]:
# Krok 1: Import knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## Krok 2 & 3: Načtení dat a rozdělení 70/30

In [ ]:
df = pd.read_csv('data/diabetes_scaled.csv')
features = [c for c in df.columns if c != 'outcome']
X = df[features].values
y = df['outcome'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Trénovací sada: {X_train.shape[0]} pacientek")
print(f"Testovací sada:  {X_test.shape[0]} pacientek")


## Krok 4 & 5: Definice Hyperopt prostoru a Objective funkce
Hledáme optimální kombinaci typu jádra (`linear`, `rbf`, `poly`, `sigmoid`), inverzní regularizace `C` a parametru jádra `gamma`.

In [ ]:
kernel_list = ['linear', 'rbf', 'poly', 'sigmoid']

space = {
    'kernel': hp.choice('kernel', kernel_list),
    'C': hp.loguniform('C', np.log(1e-3), np.log(1e3)),
    'gamma': hp.loguniform('gamma', np.log(1e-4), np.log(1e1)),
    'degree': hp.choice('degree', [2, 3])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(params):
    clf = SVC(
        kernel=params['kernel'],
        C=params['C'],
        gamma=params['gamma'],
        degree=params['degree'],
        probability=True,
        random_state=42
    )
    # Maximalizujeme F1-Score (stejně jako v cvičení 1 a 2)
    scores = cross_val_score(clf, X_train, y_train, scoring='f1', cv=cv, n_jobs=-1)
    loss = -scores.mean()
    return {'loss': loss, 'status': STATUS_OK, 'f1': scores.mean()}


## Krok 6: Bayesovská optimalizace (TPE)

In [ ]:
trials = Trials()

best_raw = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials,
    rstate=np.random.default_rng(42)
)

best_params = {
    'kernel': kernel_list[best_raw['kernel']],
    'C': best_raw['C'],
    'gamma': best_raw['gamma'],
    'degree': [2, 3][best_raw['degree']]
}

print(f"Nejlepší F1 skóre z křížové validace: {-trials.best_trial['result']['loss']:.4f}")
print("Optimální parametry SVM:", best_params)


## Krok 7 & 8: Trénování a testování na testovací sadě

In [ ]:
best_svm = SVC(
    kernel=best_params['kernel'],
    C=best_params['C'],
    gamma=best_params['gamma'],
    degree=best_params['degree'],
    probability=True,
    random_state=42
)
best_svm.fit(X_train, y_train)

y_pred = best_svm.predict(X_test)
y_prob = best_svm.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=" * 60)
print("VÝSLEDKY SVM NA TESTOVACÍ SADĚ (70/30)")
print("=" * 60)
print(f"Accuracy:  {acc * 100:.2f} %")
print(f"Recall:    {rec * 100:.2f} %")
print(f"Precision: {prec * 100:.2f} %")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")
print(f"Počet podpůrných vektorů: {sum(best_svm.n_support_)}")
print("=" * 60)


## Krok 9: Který ze tří modelů dosáhl nejlepšího výkonu?
Porovnáme všechny 3 algoritmy vytrénované na identických datech se stejnou normalizací a spliterm 70/30.

In [ ]:
summary_df = pd.DataFrame([
    {"Model": "k-NN (k=17, Euclidean)", "Accuracy": "74.89 %", "Recall": "55.56 %", "Precision": "67.16 %", "F1-Score": 0.6081, "ROC-AUC": 0.8271, "Zachyceno TP": "45 / 81"},
    {"Model": "Logistic Regression (C=0.51, L1)", "Accuracy": "73.59 %", "Recall": "53.09 %", "Precision": "65.15 %", "F1-Score": 0.5850, "ROC-AUC": 0.8345, "Zachyceno TP": "43 / 81"},
    {"Model": f"SVM ({best_params['kernel'].upper()}, C={best_params['C']:.2f})", "Accuracy": f"{acc*100:.2f} %", "Recall": f"{rec*100:.2f} %", "Precision": f"{prec*100:.2f} %", "F1-Score": round(f1, 4), "ROC-AUC": round(auc, 4), "Zachyceno TP": f"{sum((y_test==1)&(y_pred==1))} / 81"}
])

summary_df
